In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GoldLayer") \
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.1.0"
    ) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [10]:
silver_df = spark.readStream.format("delta").load("../silverLayer/data/silver/trades")


In [ ]:
from pyspark.sql.functions import col, window,sum,count,avg,expr, round
watermarked_df = silver_df.withWatermark("trade_timestamp","10 minutes")
watermarked_df = watermarked_df.select("trade_id","trader_id","quantity","price","trade_timestamp")

In [ ]:
gold_df = (watermarked_df
           .groupBy(window("trade_timestamp","15 minutes"), col("trader_id"))
           .agg(
                sum("quantity").alias('total_quantity'),
                count("trade_id").alias('trade_count'),
                round(avg("price"), 2).alias('avg_price'),
                round(sum(expr("price * quantity")),2).alias("total_trade_value")
))

In [ ]:
"""query = (gold_df.writeStream.format("delta").outputMode("append")
         .option("checkpointLocation", "checkpoints/gold/trader_metrics")
         .option("path", "data/gold/trader_metrics")
         .queryName("gold_trader_metrics")
         .start())
spark.streams.awaitAnyTermination()"""

In [ ]:
query = (
    gold_df.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .start()
)
spark.streams.awaitAnyTermination()

In [11]:
silver_df.filter(col("trader_id").isNull()).writeStream \
    .format("console") \
    .outputMode("append") \
    .start()
spark.streams.awaitAnyTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+---------+------+-----+--------+---------------+-------------------+--------+----+------+------+-------+-----+---------+------+---------------+--------------+
|trade_id|trader_id|symbol|price|quantity|trade_timestamp|ingestion_timestamp|exchange|side|broker|source|version|topic|partition|offset|kafka_timestamp|ingestion_time|
+--------+---------+------+-----+--------+---------------+-------------------+--------+----+------+------+-------+-----+---------+------+---------------+--------------+
+--------+---------+------+-----+--------+---------------+-------------------+--------+----+------+------+-------+-----+---------+------+---------------+--------------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Users/saileshpola/.pyenv/versions/3.10.13/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

-------------------------------------------
Batch: 1
-------------------------------------------
+--------+---------+------+-----+--------+---------------+-------------------+--------+----+------+------+-------+-----+---------+------+---------------+--------------+
|trade_id|trader_id|symbol|price|quantity|trade_timestamp|ingestion_timestamp|exchange|side|broker|source|version|topic|partition|offset|kafka_timestamp|ingestion_time|
+--------+---------+------+-----+--------+---------------+-------------------+--------+----+------+------+-------+-----+---------+------+---------------+--------------+
+--------+---------+------+-----+--------+---------------+-------------------+--------+----+------+------+-------+-----+---------+------+---------------+--------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+--------+---------+------+-----+--------+---------------+-------------------+--------+----+------+------+-------+-----+---------

-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+---------+--------------+-----------+---------+-----------------+
|window                                    |trader_id|total_quantity|trade_count|avg_price|total_trade_value|
+------------------------------------------+---------+--------------+-----------+---------+-----------------+
|{2026-06-20 10:30:00, 2026-06-20 10:45:00}|T0376    |898           |1          |895.43   |804096.14        |
|{2026-06-20 10:30:00, 2026-06-20 10:45:00}|T0360    |701           |1          |594.19   |416527.19        |
|{2026-06-20 10:30:00, 2026-06-20 10:45:00}|T0009    |30030         |58         |591.89   |1.837565025E7    |
|{2026-06-20 10:30:00, 2026-06-20 10:45:00}|T0343    |495           |1          |121.22   |60003.9          |
|{2026-06-20 10:30:00, 2026-06-20 10:45:00}|T0220    |257           |1          |835.88   |214821.16        |
|{2026-06-20 10:30:00, 

...

-------------------------------------------
Batch: 8
-------------------------------------------
+------------------------------------------+---------+--------------+-----------+---------+-----------------+
|window                                    |trader_id|total_quantity|trade_count|avg_price|total_trade_value|
+------------------------------------------+---------+--------------+-----------+---------+-----------------+
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0698    |1154          |2          |461.54   |444900.56        |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0215    |700           |3          |394.06   |276193.51        |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0577    |711           |1          |475.67   |338201.37        |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0111    |950           |1          |467.95   |444552.5         |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0980    |674           |1          |529.81   |357091.94        |
|{2026-06-20 10:45:00, 

-------------------------------------------
Batch: 9
-------------------------------------------
+------------------------------------------+---------+--------------+-----------+---------+-----------------+
|window                                    |trader_id|total_quantity|trade_count|avg_price|total_trade_value|
+------------------------------------------+---------+--------------+-----------+---------+-----------------+
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0698    |1154          |2          |461.54   |444900.56        |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0215    |700           |3          |394.06   |276193.51        |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0577    |711           |1          |475.67   |338201.37        |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0111    |950           |1          |467.95   |444552.5         |
|{2026-06-20 10:45:00, 2026-06-20 11:00:00}|T0980    |674           |1          |529.81   |357091.94        |
|{2026-06-20 10:45:00, 

-------------------------------------------
Batch: 9
-------------------------------------------
+------+---------+--------------+-----------+---------+-----------------+
|window|trader_id|total_quantity|trade_count|avg_price|total_trade_value|
+------+---------+--------------+-----------+---------+-----------------+
+------+---------+--------------+-----------+---------+-----------------+



-------------------------------------------
Batch: 10
-------------------------------------------
+------+---------+--------------+-----------+---------+-----------------+
|window|trader_id|total_quantity|trade_count|avg_price|total_trade_value|
+------+---------+--------------+-----------+---------+-----------------+
+------+---------+--------------+-----------+---------+-----------------+



26/06/21 02:57:49 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE